### Data Sources, URLs, and Parameters

- **Source**: Yahoo Finance via `yfinance` package  
- **Ticker**: `^GSPC` (S&P 500 Index)  
- **Parameters**: `period='max'` to download the longest available history  
- **Data Format**: Open, High, Low, Close, Volume, Dividends, Stock Splits  

We assume that Yahoo Finance data is consistent with daily market closes and covers all available historical data.

### Assumptions & Risks

- **Assumptions**
  - Yahoo Finance provides complete and accurate historical data for the S&P 500.
  - The schema (column names: Open, High, Low, Close, Volume) remains stable.
  - Daily frequency is sufficient for downstream analysis.

- **Risks**
  - Source website or API structure may change, breaking the pipeline.
  - Occasional missing values or incorrect entries.
  - Data refresh delays during weekends or holidays.
  - Service outages or API throttling can interrupt ingestion.

## *Part-1: API Pull*

In [44]:
# importing required libraries

import yfinance as yf
import pandas as pd

In [45]:
# downloading the historical data

ticker_data = yf.Ticker('^GSPC').history(period = 'max').reset_index()

In [46]:
# checking the type of data downloaded

type(ticker_data)

pandas.core.frame.DataFrame

In [47]:
# parsing the datatypes for each columns

ticker_data.dtypes

,0
Date,"datetime64[ns, America/New_York]"
Open,float64
High,float64
Low,float64
Close,float64
Volume,int64
Dividends,float64
Stock Splits,float64


In [48]:
# checking for missing values

ticker_data.isnull().sum()

,0
Date,0
Open,0
High,0
Low,0
Close,0
Volume,0
Dividends,0
Stock Splits,0


In [49]:
# checking shape of dataframe

ticker_data.shape

(24522, 8)

In [50]:
# suppose for the purpose of the project the required columns are: date, close, volume, and stock splits, then checking for their presence:

req_cols = ['Date', 'Close', 'Volume', 'Stock Splits']

# below code should produce an error in case of one or more columns missing
# assert all(column in ticker_data.drop(columns = ['Date'], axis = 1).columns for column in req_cols) --> this should produce an error since the date column is required and we are dropping it.
assert all(column in ticker_data.columns for column in req_cols)

In [51]:
# checking if the directory exists, and if not, creating it and saving the raw data.

import os

raw_path = os.path.join('..', 'data','raw')

if raw_path not in os.listdir(os.getcwd()):
    os.makedirs(raw_path, exist_ok = True)

ticker_data[req_cols].to_csv(os.path.join(raw_path, f"api_yfinance_gspc_{str(datetime.now())}.csv"))

## *Part-2: Scraping small data table*

In [52]:
!pip install bs4

In [53]:
!pip install html-table-parser-python3

In [54]:
# import the necessary libraries

import numpy as np
import os
import requests
import pandas as pd

from bs4 import BeautifulSoup
from html_table_parser import HTMLTableParser

In [55]:
# sending request using Requests and getting the response

headers = {
    'Host': 'www.worldometers.info',
    # 'Cookie': '_ga_ZDP3BFSX60=GS2.1.s1755444470' + os.getenv('o1', '') + os.getenv('g1', '') + os.getenv('t1755444966', '') + os.getenv('j60', '') + os.getenv('l0', '') + os.getenv('h0', '') + '; _ga=GA1.1.897881857.1755444471; _li_dcdm_c=.worldometers.info; _lc2_fpi=514dc520e743--01k2wa9w2chpkde2kafcrd5r5e; _lc2_fpi_meta=%7B%22w%22%3A1755444473933%7D; gamera_user_id=b16eea62-445d-4c85-ada8-4deac8350401; ccuid=454f6f02-7c8c-4f2c-b5e6-82ece9bb80db; _cc_id=264c86cbee3c06aec959e8f5bd65c5fc; panoramaId_expiry=1756049274213; panoramaId=4f1e8743a35fdccf73ec23e722224945a70285e547e351063be57307f551f3c0; panoramaIdType=panoIndiv; csuuidSekindo=68a1f4fa6ebc7; _iiq_fdata=%7B%22pcid%22%3A%224dbdff7c-e23d-f8dc-fed8-ec8003619d8a%22%2C%22pcidDate%22%3A1755444474767%2C%22gdprString%22%3A%22%22%2C%22gppString%22%3A%22%22%2C%22uspString%22%3A%22%22%2C%22gpcValue%22%3Afalse%2C%22sCal%22%3A1755444475111%2C%22isOptedOut%22%3Afalse%2C%22dbsaved%22%3A%22false%22%2C%22group%22%3A%22B%22%7D; pbjs_fabrickId=%7B%22fabrickId%22%3A%22E1%3Auit_ENOGq6-LZCIN4E-KHZI0epXnuF9e4jO3q_JvU1F9PrtDeYLOU5-nZcOo_g3dCTvZZ9cz3R-_Dp5QEMXcBWATDmVDYBOuYeroio0T5hg%22%7D; pbjs_fabrickId_cst=zix7LPQsHA%3D%3D; _lr_retry_request=true; _lr_env_src_ats=false; __gads=ID=895fe03a5431da2f:T=1755444474:RT=1755444801:S=ALNI_MZAGYxN-f9NTxDGPClyzEhvGqkASw; __gpi=UID=000011044c5877b0:T=1755444474:RT=1755444801:S=ALNI_MYbFn17qWVsoXj9RlHksLagPI3Pbg; __eoi=ID=727160af9e7d88cd:T=1755444474:RT=1755444801:S=AA-Afjbxs5SD_nnPmte2C1w3W2eH; _au_1d=AU1D-0100-001755444475-SO4GL872-HYBU; __qca=P1-ab3ff23c-7f7c-4347-b4b3-1495bd422f6e; pbjs-unifiedid=%7B%22TDID%22%3A%222dcbdf99-ddbd-456d-bc6d-b785909f66d5%22%2C%22TDID_LOOKUP%22%3A%22TRUE%22%2C%22TDID_CREATED_AT%22%3A%222025-07-17T15%3A27%3A55%22%7D; pbjs-unifiedid_cst=YiwPLDosoA%3D%3D; pbjs-unifiedid_last=Sun%2C%2017%20Aug%202025%2015%3A35%3A57%20GMT; cto_bundle=TkXAa19zcWtTZUp5SG9UVW1vMm1hS0RvZnFRaWY2b0k5TTFBWmFEeTRNcWs5MWVneUswYzBPSTJhNSUyRmgyJTJCZXNlTFNKRTVEVDVMSXd2bUlXakxvM2pIZXJ0YmNIbjJpVXhJMCUyRnZ2b21FR2UlMkZnd3c4MmZBZUtMUXhTazk0R00xMG12QXVEb0tEaHltTWNlcHpXY3ZVUW8wZ1o3QWMlMkJZMSUyQnl4cUI1RVBReWh2ZDVsSnAxc3pqTjUwcERtVWtncWFyZ1lKb3Q; _ga_FVWZ0RM4DH=GS2.1.s1755444513' + os.getenv('o1', '') + os.getenv('g1', '') + os.getenv('t1755444939', '') + os.getenv('j60', '') + os.getenv('l0', '') + os.getenv('h0', '') + '; FCNEC=%5B%5B%22AKsRol-sMRjQFWEfMABCOtZG3cSAKBhnaLxb-070aeZV-krpZAkeLOAWOhFvr48U3ETaCxsvzk-k7Mo71BxCR3W0nvmL91Zbiy2NVvfktl3QTBmcCUpuFoqO78AsmSUs7dq1RZyGl4TLTLmAcHx9gE4l-5slDUW73A%3D%3D%22%5D%5D; ccsid=d4f5cb5e-952d-477f-926f-c69543397e55',
    'user-agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:141.0) Gecko/20100101 Firefox/141.0',
    'accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
    'accept-language': 'en-US,en;q=0.5',
    'upgrade-insecure-requests': '1',
    'sec-fetch-dest': 'document',
    'sec-fetch-mode': 'navigate',
    'sec-fetch-site': 'none',
    'sec-fetch-user': '?1',
    'priority': 'u=0, i',
    # Requests doesn't support trailers
    # 'te': 'trailers',
}

response = requests.get('https://www.worldometers.info/coronavirus/', headers=headers) # cookies=cookies)

In [56]:
# creating beautiful soup object and parsing the table content using HTMLTableParser

html_soup = BeautifulSoup(response.content)
table_html = html_soup.find({'table':'main_table_countries_today'})

table_parser = HTMLTableParser()
table_parser.feed(str(table_html))
table_data = table_parser.tables[0]

In [57]:
# removing extra columns and equating empty spaces to NaN values

covid_data = pd.DataFrame(data = table_data[9:], columns = table_data[0])
covid_data = covid_data.drop(index = covid_data[covid_data['Country, Other'] == 'Total:'].index).drop(columns = ['#'])

for column in covid_data.columns:
  covid_data[column] = covid_data[column].str.replace(',','').str.replace('N/A','').replace('',np.nan)

/tmp/ipython-input-684269276.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  covid_data[column] = covid_data[column].str.replace(',','').str.replace('N/A','').replace('',np.nan)


In [58]:
# checking the datatypes of all the columns

covid_data.dtypes # --> as the results below show many of the columns which we expect to be int/float are shown as objects! Correcting it!

,0
"Country, Other",object
Total Cases,object
New Cases,float64
Total Deaths,object
New Deaths,float64
Total Recovered,object
New Recovered,object
Active Cases,object
"Serious, Critical",object
Tot Cases/ 1M pop,object


In [59]:
# converting object type columns to float type columns (as they are incorrect)

num_cols = [column for column in covid_data.columns if covid_data[column].dtype == 'object' and column not in ['Country, Other', 'Continent']]
covid_data[num_cols] = covid_data[num_cols].apply(pd.to_numeric, errors='coerce')

In [60]:
# checking the datatypes again

covid_data.dtypes # --> as expected the results datatypes make sense now!

,0
"Country, Other",object
Total Cases,int64
New Cases,float64
Total Deaths,float64
New Deaths,float64
Total Recovered,float64
New Recovered,float64
Active Cases,float64
"Serious, Critical",float64
Tot Cases/ 1M pop,float64


In [61]:
# checking the column wise null-count

covid_data.isnull().sum() # --> as we expected, the results show presence of one or more np.nan for almost every column.

,0
"Country, Other",0
Total Cases,0
New Cases,231
Total Deaths,5
New Deaths,231
Total Recovered,48
New Recovered,226
Active Cases,47
"Serious, Critical",179
Tot Cases/ 1M pop,2


In [62]:
# YYYYMMDD-HHMM
from datetime import datetime



In [63]:
# checking if the directory exists, and if not, creating it and saving the raw data.

import os

raw_path = os.path.join('..', 'data','raw')

if raw_path not in os.listdir(os.getcwd()):
    os.makedirs(raw_path, exist_ok = True)

covid_data.to_csv(os.path.join(raw_path, f"scrape_worldometers_covidinfo_{str(datetime.now())}.csv"))